# Conditional Strategies II: Timing Expected Returns

The ideas are in the slides. Here you test the most famous timing signal for the market, the dividend yield: in sample,
then the honest way, out of sample. Then the same test for four factors, each with its own price signal, the value
spread. Then a search over 24 more signals, and how many survive. Last, the other market signals from the slides:
one at a time, then combined.

**The plan**, one section per exercise in the slides

1. The dividend yield, at three horizons
2. 🎯 Out of sample: your turn
3. The timing strategy
4. 🎯 Each factor's value spread: your turn
5. Timing the six factors: 24 tries
6. Pick a predictor: any of the signals from the slides
7. Combine the signals: principal components, and the average forecast. Then back to the slides
8. 🎯 Challenge (homework): one number and three sentences

> 🧩 **Every section stands alone.** Run the Setup cell once, then jump to any section, in any order. Within a section,
> run the cells from the top.

📑 **Slides for this class:** [Conditional Strategies II (PDF)](https://amoreira2.github.io/UG54/slides/conditional_strategies_2.pdf)

<iframe src="../../slides/conditional_strategies_2.pdf" width="100%" height="520" style="border: 1px solid #ddd;"></iframe>

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

crsp = pd.read_csv(f"{BASE}/Markettiming_data1.csv", parse_dates=['date'], index_col='date')   # CRSP market, with dividends
crsp.index = crsp.index + pd.offsets.MonthEnd(0)
rf = pd.read_parquet(f"{BASE}/factors_monthly_1926.parquet")['RF']                              # T-bill rate
V  = pd.read_parquet(f"{BASE}/value_spreads_monthly_1926.parquet")    # four factors, and the book-to-market of their legs
ff = pd.read_csv(f"{BASE}/ff_monthly.csv", index_col=0, parse_dates=True)                       # the six factors, 1980-2026
F  = ff[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']].dropna()

C = crsp.join(rf, how='inner').dropna()
C['ex'] = C['vwretd'] - C['RF']                             # the market's excess return
C['dp'] = C['dp'].rolling(12).sum()                         # dividend yield: last 12 months of dividends over today's price
C['lx'] = np.log1p(C['vwretd']) - np.log1p(C['RF'])         # log excess return: these add up across months

# the other signals from the slides, for sections 6 and 7: Goyal and Welch's data, updated to 2025, and VIX
gw  = pd.read_csv(f"{BASE}/goyal_welch_monthly_2025.csv", parse_dates=['date'], index_col='date')
vix = pd.read_csv(f"{BASE}/vix_monthly_fred.csv", parse_dates=['date'], index_col='date')['vix']   # last close of each month
cpi = (1 + gw['infl']).cumprod()                             # the price level, from monthly CPI inflation
real_e10 = (gw['E12'] / cpi).rolling(120).mean()             # ten years of S&P 500 earnings, in today's dollars
predictors = pd.DataFrame({
    'D/P':               C['dp'],                            # dividend yield, as above
    'E/P':               gw['E12'] / gw['Index'],            # S&P 500 earnings over the last 12 months, over price
    '1/CAPE':            real_e10 / (gw['Index'] / cpi),     # Shiller's CAPE, turned into a yield like D/P
    'book-to-market':    gw['b/m'],                          # Dow Jones Industrials: book value over market value
    'short rate':        gw['tbl'],                          # 3-month T-bill yield
    'term spread':       gw['lty'] - gw['tbl'],              # long-term government bond yield minus the T-bill
    'default spread':    gw['BAA'] - gw['AAA'],              # Baa minus Aaa corporate bond yields
    'VIX':               vix,                                # option-implied volatility, from 1990
    'realized variance': gw['svar'],                         # sum of squared daily S&P 500 returns in the month
    'net issuance':      gw['ntis'],                         # net equity issued by NYSE firms over 12 months, over their value
}).reindex(C.index).loc['1927':]                             # one row per month, lined up with the market

sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)

print(f"market {C.index[0]:%Y-%m} to {C.index[-1]:%Y-%m}; value spreads {V.index[0]:%Y-%m} to {V.index[-1]:%Y-%m}; "
      f"six factors {F.index[0]:%Y-%m} to {F.index[-1]:%Y-%m}; {predictors.shape[1]} other signals to {predictors.index[-1]:%Y-%m}")

---

## 1. The dividend yield, at three horizons

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

When prices are low relative to dividends, either dividends are about to fall, or future returns are high. First compare
the dividend yield with the return over the *next* five years.

In [ ]:
next5 = C['lx'].rolling(60).sum().shift(-60) / 5            # return over the next 60 months, a year

ax = C['dp'].plot(label='dividend yield', legend=True)
next5.plot(ax=ax, secondary_y=True, label='return over the next five years, a year', legend=True)
plt.show()

print(f"correlation: {C['dp'].corr(next5):.2f}")

Now the forecasting regression,

$$r_{t \to t+h} = a + b \cdot \text{D/P}_t + u,$$

run for the next month, the next year and the next five years. Two sets of $t$-statistics: the naive ones, and HAC ones
that allow for the overlap.

In [ ]:
rows = {}
for h in [1, 12, 60]:
    y = C['lx'].rolling(h).sum().shift(-h) * 12 / h        # return over the next h months, a year
    data = pd.DataFrame({'y': y, 'dp': C['dp']}).loc['1927':].dropna()
    X = sm.add_constant(data['dp'])

    naive = sm.OLS(data['y'], X).fit()                                                  # as if every month were new
    hac   = sm.OLS(data['y'], X).fit(cov_type='HAC', cov_kwds={'maxlags': max(h, 1)})  # allows for the overlap

    rows[f"{h} months"] = {'b': naive.params['dp'], 'naive t': naive.tvalues['dp'], 'HAC t': hac.tvalues['dp'],
                           'R2': naive.rsquared, 'observations': len(data), 'do not overlap': len(data) // h}
print(pd.DataFrame(rows).T.round(3).to_string())

- When was the dividend yield highest? What came next?
- The $R^2$ and the naive $t$ both grow with the horizon. Should you believe the five-year regression most?
- Two consecutive five-year returns share 59 of their 60 months. How many truly separate five-year periods are there?
  Which $t$-statistic would you report?

---

## 2. 🎯 Out of sample

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

The honest test: stand in each month $t$, using only what you knew then.

- fit the regression of next month's return on D/P, using only months whose next return you have already seen
- forecast next month's return with today's D/P
- the benchmark: the historical average return up to $t$, which is what you would use with no signal

The loop below does this every month from 1950. Read it before you run it.

In [ ]:
y = C['ex'].shift(-1)                      # next month's excess return, lined up with this month's D/P
rows = []
for t in C.loc['1950':].index[:-1]:
    known = pd.DataFrame({'y': y, 'dp': C['dp']}).loc['1927':t].iloc[:-1].dropna()   # pairs whose return is known at t
    b, a = np.polyfit(known['dp'], known['y'], 1)          # slope and intercept, from the past only
    rows.append({'date': t,
                 'r':        y.loc[t],                     # what happened next month
                 'forecast': a + b * C['dp'].loc[t],       # what D/P said
                 'average':  known['y'].mean()})           # what the historical average said
P = pd.DataFrame(rows).set_index('date')
P.head()

**Your turn.** The out-of-sample $R^2$ compares the two forecasts' squared errors:

$$R^2_{\text{OOS}} = 1 - \frac{\sum (r - \text{forecast})^2}{\sum (r - \text{average})^2}$$

Positive: D/P beat the historical average. Negative: you would have done better without it. Call it `r2_oos`.

In [ ]:
# === YOUR TURN ===

r2_oos = ____

In [ ]:
#@title 🔒 Check: out of sample
r2 = lambda Q: 1 - ((Q['r'] - Q['forecast']) ** 2).sum() / ((Q['r'] - Q['average']) ** 2).sum()

try:
    print(f"yours: {r2_oos:.2%}")
except Exception as e:
    print("yours: not built yet:", type(e).__name__)

print(f"out of sample R2, 1950-2024: {r2(P):.2%}")
for a, b in [('1950', '1979'), ('1980', '1999'), ('2000', '2024')]:
    print(f"   {a}-{b}: {r2(P.loc[a:b]):.2%}")

gain = ((P['r'] - P['average']) ** 2 - (P['r'] - P['forecast']) ** 2).cumsum()   # rising when D/P forecasts better
gain.plot(title='cumulative gain from D/P over the historical average'); plt.axhline(0, color='k', lw=0.5); plt.show()

- When did D/P stop working? Look at the plot.
- In sample, D/P explained 0.3% of next month's return. Out of sample it explains less than nothing. How can both be true?

---

## 3. The timing strategy

> 🧩 **Stands alone.** Needs only the Setup cell: the first cell below rebuilds section 2's forecasts.

The formula from *Capital Allocation I* with a forecast in it: weight $= \text{forecast} / (\gamma \sigma^{2})$. We hold
$\sigma$ fixed at its value before 1950, so all the timing comes from the forecast.

In [ ]:
#@title Section 2's forecasts, rebuilt here so this section stands alone
y = C['ex'].shift(-1)                      # next month's excess return, lined up with this month's D/P
rows = []
for t in C.loc['1950':].index[:-1]:
    known = pd.DataFrame({'y': y, 'dp': C['dp']}).loc['1927':t].iloc[:-1].dropna()   # pairs whose return is known at t
    b, a = np.polyfit(known['dp'], known['y'], 1)
    rows.append({'date': t, 'r': y.loc[t], 'forecast': a + b * C['dp'].loc[t], 'average': known['y'].mean()})
P = pd.DataFrame(rows).set_index('date')
print(f"D/P's forecasts for {len(P)} months, {P.index[0]:%Y-%m} to {P.index[-1]:%Y-%m}")

In [ ]:
sigma2 = C['ex'].loc[:'1949'].var()        # the market's monthly variance before 1950, held fixed
gamma  = 3
weight = P['forecast'] / (gamma * sigma2)  # the weight on the market each month: negative means short
timed  = weight * P['r']                   # this month's excess return on that weight

print(f"timing: Sharpe {sharpe(timed):.2f}    buy and hold: Sharpe {sharpe(P['r']):.2f}")
print(f"average weight {weight.mean():.2f}, short in {np.mean(weight < 0):.0%} of months\n")
for a, b in [('1950', '1979'), ('1980', '1999'), ('2000', '2024')]:
    print(f"{a}-{b}: timing {sharpe(timed.loc[a:b]):.2f}, buy and hold {sharpe(P['r'].loc[a:b]):.2f}")

weight.plot(title='weight on the market'); plt.axhline(0, color='k', lw=0.5); plt.show()

- Would changing `gamma` change the Sharpe ratio? Try it.
- In which period did timing win, and in which did it lose badly? Why then?

---

## 4. 🎯 Each factor's value spread

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

D/P is the market's price against its fundamentals. A long-short factor has one too: how cheap the stocks it buys are
against the stocks it sells,

$$\text{VS}_t = \log \frac{B}{M}\Big|_{\text{long},\,t} - \log \frac{B}{M}\Big|_{\text{short},\,t}.$$

`V` has four factors from Ken French's 2×3 sorts: each one's monthly return (`HML`, `SMB`, `RMW`, `CMA`) and the
book-to-market of its long and short legs (`HML_bm_long`, `HML_bm_short`, and so on). HML and SMB start in 1926, RMW and
CMA in 1963. Book is from the last fiscal year; the price is from last December.

**Your turn.** Build HML's value spread and call it `vs_hml`. `np.log` takes logs.

In [ ]:
# === YOUR TURN ===

vs_hml = ____

In [ ]:
#@title 🔒 Check: HML's value spread
vs_ref = np.log(V['HML_bm_long']) - np.log(V['HML_bm_short'])

try:
    print("yours matches" if np.allclose(vs_hml, vs_ref) else "yours differs: compare with the plot below", "\n")
except Exception as e:
    print("yours: not built yet:", type(e).__name__, "\n")

next5_hml = V['HML'].rolling(60).sum().shift(-60) / 5       # HML's return over the next five years, a year
ax = vs_ref.plot(label="HML's value spread", legend=True)
next5_hml.plot(ax=ax, secondary_y=True, label='HML over the next five years, a year', legend=True)
plt.show()

print(f"median {vs_ref.median():.2f}, now {vs_ref.iloc[-1]:.2f} ({vs_ref.index[-1]:%Y-%m}); "
      f"correlation with the next five years: {vs_ref.corr(next5_hml):.2f}")

Now sections 1 to 3 again, in a loop over the four factors: the in-sample regression on next month's return, the
out-of-sample loop with forecasts from 1980, and the timing strategy with $\sigma$ fixed at its value before 1980.

In [ ]:
rows = {}
for f in ['HML', 'SMB', 'RMW', 'CMA']:
    vs = (np.log(V[f'{f}_bm_long']) - np.log(V[f'{f}_bm_short'])).dropna()   # this factor's value spread
    ret_next = V[f].shift(-1)                              # next month's return, lined up with this month's spread
    data = pd.DataFrame({'y': ret_next, 'vs': vs}).dropna()

    # in sample, as in section 1
    fit = sm.OLS(data['y'], sm.add_constant(data['vs'])).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

    # out of sample from 1980, as in section 2
    oos = []
    for t in vs.loc['1980':].index[:-1]:
        known = data.loc[:t].iloc[:-1]                     # pairs whose return is known at t
        b, a = np.polyfit(known['vs'], known['y'], 1)
        oos.append({'date': t, 'r': ret_next.loc[t], 'forecast': a + b * vs.loc[t], 'average': known['y'].mean()})
    Q = pd.DataFrame(oos).set_index('date')

    # the timing strategy, as in section 3
    weight = Q['forecast'] / (3 * V[f].loc[:'1979'].var())
    timed  = weight * Q['r']

    rows[f] = {'from': vs.index[0].year, 'HAC t': fit.tvalues['vs'], 'R2 in sample': fit.rsquared,
               'R2 out of sample': 1 - ((Q['r'] - Q['forecast']) ** 2).sum() / ((Q['r'] - Q['average']) ** 2).sum(),
               'Sharpe, buy and hold': sharpe(Q['r']), 'Sharpe, timed': sharpe(timed), 'months short': np.mean(weight < 0)}

print(pd.DataFrame(rows).T.round(3).to_string())

- Which factors beat the historical average out of sample? Set them against D/P's $R^2$ in section 2.
- Where is the gain largest? How often is that strategy short, and would you run it?
- The price in the book-to-market is last December's, not today's. What has the factor done since December, and why
  might that help the forecast?

---

## 5. Timing the six factors: 24 tries

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

Four simple signals for each factor: its own return last month, over the last 12 months, three to five years ago, and its
own volatility. Run the forecasting regression on 1980–2002 and again on 2003–2026. A real signal should work in both.

In [ ]:
d   = pd.read_parquet(f"{BASE}/ff6_daily_1963.parquet")                 # daily factors, for each factor's volatility
own_vol = d[F.columns].groupby(d.index + pd.offsets.MonthEnd(0)).apply(lambda x: np.sqrt(((x - x.mean()) ** 2).sum()))  # each month's volatility

rows = []
for f in F.columns:
    r = F[f]
    signals = {'last month':     r,
               'last 12 months': r.rolling(12).sum(),
               '3-5 years ago':  r.shift(12).rolling(48).sum(),
               'own volatility': own_vol[f].reindex(r.index)}
    for name, s in signals.items():
        data = pd.DataFrame({'y': r.shift(-1), 'x': s}).dropna()       # next month's return on this month's signal
        row = {'factor': f, 'signal': name}
        for half, sl in [('t, 1980-2002', slice('1980', '2002')), ('t, 2003-2026', slice('2003', '2026'))]:
            q = data.loc[sl]
            row[half] = sm.OLS(q['y'], sm.add_constant(q['x'])).fit().tvalues['x']
        rows.append(row)
T = pd.DataFrame(rows)
print(T.round(2).sort_values('t, 1980-2002', ascending=False).to_string(index=False))

plt.scatter(T['t, 1980-2002'], T['t, 2003-2026'])
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
plt.xlabel('t, 1980-2002'); plt.ylabel('t, 2003-2026'); plt.show()

- How many of the 24 have $|t| > 2$ in the first half? How many would you expect by luck alone?
- Take the five best from the first half. How many still work in the second?
- Which signal works for several factors in both halves? It has a name: factor momentum.

---

## 6. Pick a predictor

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

The slides list eleven signals people use to forecast the market. `predictors` has ten of them, one column each, lined
up with the market month by month. Forward P/E is missing: it needs analysts' forecasts, which we do not have.

| column | what it is | from |
|---|---|---|
| `D/P` | dividends over the last 12 months, over price | 1927 |
| `E/P` | S&P 500 earnings over the last 12 months, over price | 1927 |
| `1/CAPE` | ten years of real earnings over the real price: Shiller's CAPE, as a yield | 1927 |
| `book-to-market` | book value over market value, Dow Jones Industrials | 1927 |
| `short rate` | 3-month T-bill yield | 1927 |
| `term spread` | long-term government bond yield minus the T-bill | 1927 |
| `default spread` | Baa minus Aaa corporate bond yields | 1927 |
| `VIX` | option-implied volatility of the S&P 500 | 1990 |
| `realized variance` | sum of squared daily S&P 500 returns in the month | 1927 |
| `net issuance` | net equity issued over the last 12 months, over market value | 1927 |

The valuation ratios are all yields, like D/P: a high value means a cheap market.

Set `PREDICTOR` to a column name and run the cell. It does sections 1 to 3 on that signal: the in-sample regression, the
out-of-sample loop from `START`, and the timing strategy.

In [ ]:
PREDICTOR = 'E/P'      # one of: 'D/P', 'E/P', '1/CAPE', 'book-to-market', 'short rate', 'term spread', 'default spread',
                       #         'VIX', 'realized variance', 'net issuance'
START     = '1950'     # the first month to forecast

x = predictors[PREDICTOR].dropna()
start = max(pd.Timestamp(START), x.index[0] + pd.DateOffset(years=10))   # at least ten years of data before the first forecast

# in sample, as in section 1: next month's log excess return on this month's signal
data = pd.DataFrame({'y': C['lx'].shift(-1), 'x': x}).dropna()
fit  = sm.OLS(data['y'], sm.add_constant(data['x'])).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

# out of sample, as in section 2: each month, fit on the past only
y = C['ex'].shift(-1)
pairs = pd.DataFrame({'y': y, 'x': x})
rows = []
for t in x.loc[start:].index[:-1]:
    known = pairs.loc[:t].iloc[:-1].dropna()               # pairs whose return is known at t
    b, a = np.polyfit(known['x'], known['y'], 1)
    rows.append({'date': t, 'r': y.loc[t], 'forecast': a + b * x.loc[t], 'average': known['y'].mean()})
fc = pd.DataFrame(rows).set_index('date')
r2_out = 1 - ((fc['r'] - fc['forecast']) ** 2).sum() / ((fc['r'] - fc['average']) ** 2).sum()

# the timing strategy, as in section 3, with the variance held at its value before the first forecast
weight = fc['forecast'] / (3 * C['ex'][C.index < start].var())
timed  = weight * fc['r']

print(f"{PREDICTOR}, {x.index[0]:%Y-%m} to {x.index[-1]:%Y-%m}")
print(f"in sample:  b {fit.params['x']:.3f}, HAC t {fit.tvalues['x']:.2f}, R2 {fit.rsquared:.2%}")
print(f"out of sample, forecasts from {start:%Y-%m}: R2 {r2_out:.2%}")
print(f"timing: Sharpe {sharpe(timed):.2f}, buy and hold {sharpe(fc['r']):.2f}, short in {np.mean(weight < 0):.0%} of months")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
x.plot(ax=ax[0], title=PREDICTOR)
gain = ((fc['r'] - fc['average']) ** 2 - (fc['r'] - fc['forecast']) ** 2).cumsum()   # rising when the signal forecasts better
gain.plot(ax=ax[1], title=f'cumulative gain from {PREDICTOR} over the historical average')
ax[1].axhline(0, color='k', lw=0.5)
plt.show()

- Try `E/P`, the strongest in sample, then `short rate`. Which beats the historical average out of sample?
- Try `VIX`. Its forecasts start in 2000, after ten years of data. Is that enough to estimate a slope?
- Pick a signal and set `START = '1980'`. Does the verdict change?
- Every one of these signals was published because it forecast well in some sample. Why might that make it look
  better in sample than it will do from here on?

The cell below runs the same three steps on all ten at once: the table in the slides.

In [ ]:
#@title All ten at once: the table in the slides
y = C['ex'].shift(-1)
sigma2 = C['ex'].loc[:'1949'].var()
table = {}
for name in predictors.columns:
    x = predictors[name].dropna()
    start = max(pd.Timestamp('1950'), x.index[0] + pd.DateOffset(years=10))

    data = pd.DataFrame({'y': C['lx'].shift(-1), 'x': x}).dropna()
    fit  = sm.OLS(data['y'], sm.add_constant(data['x'])).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

    pairs = pd.DataFrame({'y': y, 'x': x})
    rows = []
    for t in x.loc[start:].index[:-1]:
        known = pairs.loc[:t].iloc[:-1].dropna()
        b, a = np.polyfit(known['x'], known['y'], 1)
        rows.append({'date': t, 'r': y.loc[t], 'forecast': a + b * x.loc[t], 'average': known['y'].mean()})
    fc = pd.DataFrame(rows).set_index('date')

    table[name] = {'from': x.index[0].year, 'HAC t': fit.tvalues['x'], 'forecasts from': start.year,
                   'R2 out of sample': 1 - ((fc['r'] - fc['forecast']) ** 2).sum() / ((fc['r'] - fc['average']) ** 2).sum(),
                   'R2, 2000-2024': 1 - ((fc['r'] - fc['forecast']) ** 2).loc['2000':].sum() / ((fc['r'] - fc['average']) ** 2).loc['2000':].sum(),
                   'Sharpe, timed': sharpe(fc['forecast'] / (3 * sigma2) * fc['r']), 'Sharpe, buy and hold': sharpe(fc['r'])}

print(pd.DataFrame.from_dict(table, orient='index').round(4).to_string())

---

## 7. Combine the signals: principal components, and the average forecast

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

One at a time, almost none of the signals survives out of sample. But they overlap: the four valuation ratios say *cheap*
or *dear* at the same time. Two ways to combine them:

- **Principal components (PCA).** Standardize each signal: subtract its mean, divide by its standard deviation. The first
  principal component, PC1, is the weighted sum of the standardized signals that varies the most. PC2 is the one that
  varies the most among those uncorrelated with PC1, and so on. Then forecast with PC1 alone.
- **Average the forecasts.** Make one forecast per signal, as in section 6, and average them.

We use the nine signals that go back to 1927. VIX starts only in 1990.

In [ ]:
from sklearn.decomposition import PCA

nine = predictors.drop(columns='VIX').dropna()            # the nine signals with a history back to 1927
z = (nine - nine.mean()) / nine.std()                      # each signal in units of its own standard deviation
pca = PCA().fit(z)

print("share of the nine signals' variance in each component:")
print(pd.Series(pca.explained_variance_ratio_, index=[f'PC{i + 1}' for i in range(9)]).round(2).to_string())

weights = pd.DataFrame(pca.components_[:3].T, index=nine.columns, columns=['PC1', 'PC2', 'PC3'])   # each component's weights
weights.plot.barh(subplots=True, layout=(1, 3), figsize=(13, 3.6), legend=False, sharex=True, sharey=True)
plt.show()

- Which signals have the largest weights in PC1? In PC2? Give each component a name.

Now forecast with PC1. In sample is easy, but it uses the whole history's means, standard deviations and weights:
information nobody had in 1950. Out of sample, the loop redoes the standardizing and the PCA each month, on the past only.

In [ ]:
# in sample: PC1 computed with the whole history
pc1 = pd.Series(pca.transform(z)[:, 0], index=z.index)
data = pd.DataFrame({'y': C['lx'].shift(-1), 'pc1': pc1}).dropna()
fit  = sm.OLS(data['y'], sm.add_constant(data['pc1'])).fit(cov_type='HAC', cov_kwds={'maxlags': 1})
print(f"in sample: PC1's HAC t {fit.tvalues['pc1']:.2f}, R2 {fit.rsquared:.2%}")

# out of sample: each month, standardize and run the PCA on the past only, then forecast as in section 2
y = C['ex'].shift(-1)
rows = []
for t in nine.loc['1950':].index[:-1]:
    past = nine.loc[:t]
    z_past = (past - past.mean()) / past.std()
    pc1_past = pd.Series(PCA(n_components=1).fit_transform(z_past)[:, 0], index=past.index)
    known = pd.DataFrame({'y': y, 'pc1': pc1_past}).loc[:t].iloc[:-1].dropna()   # pairs whose return is known at t
    b, a = np.polyfit(known['pc1'], known['y'], 1)          # if the PCA flips PC1's sign, b flips too: same forecast
    rows.append({'date': t, 'r': y.loc[t], 'forecast': a + b * pc1_past.loc[t], 'average': known['y'].mean()})
fc_pca = pd.DataFrame(rows).set_index('date')
r2_pca = 1 - ((fc_pca['r'] - fc_pca['forecast']) ** 2).sum() / ((fc_pca['r'] - fc_pca['average']) ** 2).sum()
print(f"out of sample, forecasts from 1950: R2 {r2_pca:.2%}")

The other way: one forecast per signal, then the average of the nine. The table sets both combinations against each
signal alone, with the same benchmark for all: the historical average.

In [ ]:
y = C['ex'].shift(-1)
months = nine.loc['1950':].index[:-1]
forecasts = pd.DataFrame(index=months)
for name in nine.columns:                                  # one forecast per signal, as in section 6
    pairs = pd.DataFrame({'y': y, 'x': predictors[name]})
    f = []
    for t in months:
        known = pairs.loc[:t].iloc[:-1].dropna()
        b, a = np.polyfit(known['x'], known['y'], 1)
        f.append(a + b * pairs['x'].loc[t])
    forecasts[name] = f
forecasts['PC1'] = fc_pca['forecast']                      # from the cell above
forecasts['average of the nine'] = forecasts[nine.columns].mean(axis=1)

r    = y.loc[months]                                                        # what happened next month
hist = pd.Series([y.loc['1927':t].iloc[:-1].mean() for t in months], index=months)   # the historical average, known at t
sigma2 = C['ex'].loc[:'1949'].var()
table = {}
for name in forecasts.columns:
    f = forecasts[name]
    row = {}
    for a, b in [('1950', '2024'), ('1950', '1979'), ('1980', '1999'), ('2000', '2024')]:
        row[f'R2 {a}-{b}'] = 1 - ((r - f) ** 2).loc[a:b].sum() / ((r - hist) ** 2).loc[a:b].sum()
    row['Sharpe, timed'] = sharpe(f / (3 * sigma2) * r)
    table[name] = row
print(pd.DataFrame.from_dict(table, orient='index').round(4).to_string())
print(f"\nbuy and hold: Sharpe {sharpe(r):.2f}")

ax = (12 * forecasts[['D/P', 'PC1', 'average of the nine']]).plot(lw=1, title="forecast of next month's excess return, a year")
(12 * hist).plot(ax=ax, color='k', lw=1.5, label='historical average', legend=True)
plt.show()

- Does PC1 forecast better than D/P alone? PCA picks the weights that best summarize the nine signals. It never looks
  at returns. Why might that matter?
- The average of the nine forecasts beats every signal on its own. Think of each forecast's error as a risky asset:
  what does averaging nine of them do? Where have you seen this before?
- In the plot, the average forecast stays closer to the historical average than any one signal does. Why is that a
  good thing here?

---

## ⏸ Back to the slides

For the takeaways. The challenge below is homework.

---

## 🎯 Challenge: start later

> 🧩 **Stands alone.** Needs only the Setup cell, and the loop from section 2 to copy.

*Homework — due before the next class. It should take ten minutes.*

Section 2 started forecasting in 1950. Suppose you had only started in **1980**, the year the value spreads in section 4
start. Rerun the loop from section 2 with `C.loc['1980':]` instead of `C.loc['1950':]`, and everything else the same.
Report the out-of-sample $R^2$.

Then, in at most three sentences: would you trade D/P on this evidence? Set it against the value spreads in section 4.

> **📌 Required variable names:**
> ```python
> oos_r2_1980 = ____   # out-of-sample R2 with forecasts from 1980, a decimal (-0.01 means -1%)
> MEMO = """...""" # at most three sentences
> ```

In [ ]:
# Your work here


oos_r2_1980 = ____

print(f"out of sample R2 from 1980: {oos_r2_1980:.2%}")

In [ ]:
MEMO = """
Write your answer here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ['oos_r2_1980', 'MEMO']
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L15_ConditionalStrategiesII_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")